# C3 · Imputation with the mean erases variance and correlation

**Type**: Break the model · **Level**: ●●● challenge · **Time**: about 45 minutes · **extended path**

## Objective

Measure, as the share of missing values grows, how imputation with the mean, deterministic regression and
regression with noise alter the variance, correlation and regression coefficient of a variable.

## Dataset

Synthetic: 4000 rows with `x1 ~ N(0,1)` and `x2 = 0.8·x1 + 0.6·e` (true correlation 0.8, variance 1), and `y = x1 + x2 + noise`.
The true coefficient of `x2` in the regression of `y` on `x1` and `x2` is **1**. We make a share `p` of `x2` missing, at random (MCAR).

## Educational purpose

Understand that imputation is not neutral with respect to the statistics you care about: the mean **shortens the variance**
by (1 − p) and **weakens the correlation**; deterministic regression **inflates** it; only imputation with noise preserves
both.

## Task

1. Write `imputations(p, seed=0)`: it makes a fraction `p` of `x2` missing (mask `rng.random(n) < p`) and returns a dictionary with three imputed versions of `x2`: `"mean"` (`SimpleImputer`), `"regression"` (`IterativeImputer(random_state=0)` on `[x1, x2]`) and `"with_noise"` (`IterativeImputer(sample_posterior=True, random_state=0)`).
2. Write `metrics(x2_imp)`: it returns a dictionary with `var_ratio` (variance of `x2_imp` divided by the variance of the true `x2`), `corr` (correlation between `x1` and `x2_imp`) and `coef_x2` (coefficient of `x2_imp` in `LinearRegression` of `y` on `[x1, x2_imp]`).
3. For `p` in `[0.1, 0.2, 0.3, 0.4, 0.5, 0.6]` and for each method, fill the DataFrame `results` (index `(p, method)`, columns `var_ratio`, `corr`, `coef_x2`).
4. Compute `theoretical_var = 1 − p` and `theoretical_corr = sqrt(1 − p)·0.8` for the mean imputation only: they should coincide with what you measured.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.experimental import enable_iterative_imputer          # noqa: F401
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.linear_model import LinearRegression

RHO, n = 0.8, 4000
rng = np.random.default_rng(0)
x1 = rng.normal(size=n)
x2 = RHO * x1 + np.sqrt(1 - RHO ** 2) * rng.normal(size=n)
y = x1 + x2 + rng.normal(size=n)
PS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]

## Your work

In [ ]:
def imputations(p, seed=0):
    ...


def metrics(x2_imp):
    ...

In [ ]:
rows = {}
for p in PS:
    for method, x2_imp in imputations(p).items():
        rows[(p, method)] = ...
results = pd.DataFrame(rows).T
results.index.names = ["p", "method"]
results.round(3)

In [ ]:
theoretical_var = ...
theoretical_corr = ...

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='To mask: `mask = np.random.default_rng(seed).random(n) < p; x2_miss = np.where(mask, np.nan, x2)`. To impute on two columns: `IterativeImputer(...).fit_transform(np.column_stack([x1, x2_miss]))[:, 1]`. For the mean `SimpleImputer(strategy="mean")`. Coefficient: `LinearRegression().fit(np.column_stack([x1, x2_imp]), y).coef_[1]`.'):
    check(set(imputations(0.3)) == {"mean", "regression", "with_noise"}, "three imputation methods", "keys: mean, regression, with_noise")
    check(len(results) == 18 and set(results.columns) == {"var_ratio", "corr", "coef_x2"}, "18 rows and the three metrics", "6 values of p x 3 methods")
    mean_rows = results.xs("mean", level="method")
    check(np.all(np.abs(mean_rows.var_ratio.to_numpy() - (1 - np.array(PS))) < 0.06),
          "with the mean the variance drops by about a factor (1 - p)", "var_ratio of the mean must follow 1 - p")
    check(np.all(np.abs(mean_rows["corr"].to_numpy() - np.sqrt(1 - np.array(PS)) * RHO) < 0.06),
          "with the mean the correlation drops by about sqrt(1 - p)", "corr of the mean must follow sqrt(1 - p) * 0.8")
    check(close(theoretical_var, 1 - np.array(PS)) and close(theoretical_corr, np.sqrt(1 - np.array(PS)) * RHO),
          "correct theoretical formulas", "theoretical_var = 1 - p, theoretical_corr = sqrt(1 - p) * 0.8")
    check(mean_rows.coef_x2.loc[0.6] < mean_rows.coef_x2.loc[0.1] - 0.3, "the coefficient of x2 is attenuated by the mean, more and more as p grows",
          "coef_x2 with the mean must drop sharply from p=0.1 to p=0.6")
    reg = results.xs("regression", level="method")
    check(reg["corr"].loc[0.5] > RHO + 0.05, "deterministic regression INFLATES the correlation (the imputed values are an exact function of x1)",
          "with p=0.5 the correlation after deterministic regression should exceed 0.85")
    noisy = results.xs("with_noise", level="method")
    check(abs(noisy["corr"].loc[0.5] - RHO) < 0.06 and abs(noisy.var_ratio.loc[0.5] - 1) < 0.1,
          "with noise correlation and variance stay the true ones", "with p=0.5 with_noise should have corr about 0.8 and var_ratio about 1")
done()

---
The commented solution is in `solutions/C_break_the_model/C3_mean_imputation_distorts.ipynb`. Open it after you pass the check, or after you have really tried.